In [ ]:
import cv2
import os
import numpy as np
import os
from collections import defaultdict


In [ ]:

def merge_images_by_prefix(filenames, save_dir, output_prefix, img_shape, step_x, step_y, overlap, path):
    h, w, _ = img_shape
    reconstructed_img = np.zeros((h, w, 3), dtype=np.uint8)

    for filename in filenames:
        parts = filename.split('_')
        prefix = parts[0]  
        w_idx = int(parts[-2][1:])  
        h_idx = int(parts[-1][1:-4]) 
        
        img_slice = cv2.imread(os.path.join(path, filename))
        print(f"Reading file: {os.path.join(path, filename)}")

        start_x = int(w_idx * (step_x - overlap))
        start_y = int(h_idx * (step_y - overlap))
        end_x = int(start_x + min(step_x, img_slice.shape[1]))
        end_y = int(start_y + min(step_y, img_slice.shape[0]))

        slice_width = int(end_x - start_x)
        slice_height = int(end_y - start_y)
        print(f"Slice width: {slice_width}, Slice height: {slice_height}")

        slice_no_overlap = img_slice[:slice_height, :slice_width]

        if end_x > w:
            end_x = w
        if end_y > h:
            end_y = h

        slice_no_overlap = img_slice[:end_y - start_y, :end_x - start_x]

        reconstructed_img[start_y:end_y, start_x:end_x] = slice_no_overlap

    output_name = f"{output_prefix}_reconstructed.bmp"
    cv2.imwrite(os.path.join(save_dir, output_name), reconstructed_img)
    print(f"Image saved as {output_name}")


In [ ]:

def group_and_merge_by_prefix(slice_filenames, save_dir, img_shape, step_x, step_y, overlap, path):
    groups = defaultdict(list)
    
    for filename in slice_filenames:
        prefix = filename.split('_')[0]  
        groups[prefix].append(filename)
    
    # Merge images for each group
    for prefix, group_filenames in groups.items():
        merge_images_by_prefix(group_filenames, save_dir, prefix, img_shape, step_x, step_y, overlap, path)


path = r"D:\Work\codev2\code\work_code\model_code\runs\detect\predict19"
save_path = r"D:\Work\codev2\code\bbbb"
slice_filenames = []

for file in os.listdir(path):
    if file == "labels":
        continue
    if file.endswith(".bmp"): 
        slice_filenames.append(file)

img_shape = (3648 , 5472, 3)  
step_x = 3648/3
step_y =  5472/3
overlap = 70

group_and_merge_by_prefix(slice_filenames, save_path, img_shape, step_x, step_y, overlap, path)
    

In [ ]:
img = cv2.imread(r"D:\Work\codev2\data\24.09\24.09\C05 VS-W\309.bmp")
w, h , c = img.shape

print(w, h)